# 🛡️ 보험 상담 AI 에이전트 (Google Colab)

**GPT-4o + 보험다모아 엑셀 데이터 + 실시간 웹 검색 + 개인정보 이노베이션 존 시나리오** 기반 보험 상담 챗봇

### 주요 기능
| 탭 | 기능 | 설명 |
|---|---|---|
| 💬 보험 상담 | GPT-4o 채팅 | 보험 상품 추천·비교·견적 (보험다모아 공시 + 웹 검색) |
| 💳 신용점수 포트폴리오 | 신용 기반 보험 설계 | NICE/KCB 점수 → 맞춤 보험 포트폴리오 |
| 🏥 건강위험 포트폴리오 | 건강검진 위험 분석 | 로지스틱 모델(AUC 0.78) → 위험군별 상품 추천 |
| 🏆 대회 데모 | 이노베이션 존 10시나리오 | RGST·G1E·CDW·BFC 기반 정밀 언더라이팅·신용평가·위험관리 |

### 실행 순서
1. **셀 1**: 패키지 설치
2. **셀 2**: 코드 클론
3. **셀 3**: OpenAI API 키 설정
4. **셀 4**: (선택) 엑셀 데이터 업로드
5. **셀 5**: (선택) 엑셀 → 지식베이스 반영 + ChromaDB 재구축
6. **셀 6**: (선택) ChromaDB 벡터 DB 구축 (엑셀 미사용 시)
7. **셀 7**: 서버 실행 → 생성된 URL로 접속

> **필요한 것**: OpenAI API 키만 있으면 됩니다. (별도 계정/토큰 불필요)


## 셀 1 — 패키지 설치

In [ ]:
# 핵심 패키지
!pip install -q openai python-dotenv flask requests ddgs xlrd

# ChromaDB RAG (선택 — 처음 실행 시 임베딩 모델 약 443MB 다운로드)
!pip install -q chromadb sentence-transformers

print('✅ 패키지 설치 완료')

## 셀 2 — 코드 클론

In [ ]:
import os

REPO_URL    = 'https://github.com/Sdapaul/insurance-agent.git'
PROJECT_DIR = '/content/insurance-agent'

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull

os.chdir(PROJECT_DIR)
print(f'✅ 작업 디렉터리: {os.getcwd()}')

## 셀 3 — API 키 설정

### 방법 A: Colab Secrets (권장 — 키가 노트북에 저장되지 않음)
왼쪽 사이드바 **🔑 아이콘** → `OPENAI_API_KEY` 추가

### 방법 B: 직접 입력

In [ ]:
import os

# ── 방법 A: Colab Secrets에서 자동 로드 ──
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY') or ''
    FSS_API_KEY    = userdata.get('FSS_API_KEY') or ''
    print('✅ Colab Secrets에서 키 로드 시도')
except Exception:
    OPENAI_API_KEY = ''
    FSS_API_KEY    = ''

# ── 방법 B: 직접 입력 (Secrets 미사용 시 여기에 입력) ──
if not OPENAI_API_KEY:
    OPENAI_API_KEY = 'sk-...'   # ← OpenAI API 키 입력
# FSS_API_KEY = 'YOUR_FSS_KEY' # ← 선택 (연금저축보험 조회 시)

# 환경변수 및 .env 파일 생성
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
os.environ['FSS_API_KEY']    = FSS_API_KEY

with open('/content/insurance-agent/.env', 'w') as f:
    f.write(f'OPENAI_API_KEY={OPENAI_API_KEY}\n')
    f.write(f'FSS_API_KEY={FSS_API_KEY}\n')

if not OPENAI_API_KEY or OPENAI_API_KEY == 'sk-...':
    print('⚠️  OpenAI API 키를 입력하세요!')
else:
    print(f'✅ OpenAI API 키 설정 완료 ({OPENAI_API_KEY[:8]}...)')

## 셀 4 — (선택) 보험다모아 엑셀 데이터 업로드

보험다모아(e-insmarket.or.kr)에서 다운로드한 `.xls` 파일을 업로드하면 실제 공시 데이터로 추천받을 수 있습니다.  
**업로드하지 않아도** 로컬 데이터 + 웹 검색으로 동작합니다.

In [ ]:
import os
from google.colab import files

print('엑셀 파일을 업로드하세요 (취소해도 무방)...')
try:
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dest = os.path.join('/content/insurance-agent', fname)
        with open(dest, 'wb') as f:
            f.write(data)
        print(f'  ✅ 업로드: {fname} ({len(data):,} bytes)')
    if uploaded:
        cache_path = '/content/insurance-agent/data/insmarket_excel_cache.json'
        if os.path.exists(cache_path):
            os.remove(cache_path)
        print('\n✅ 엑셀 캐시 초기화 완료 — 다음 질문 시 자동 파싱됩니다.')
except Exception as e:
    print(f'업로드 건너뜀: {e}')

## 셀 5 — (선택) 엑셀 → 지식베이스 반영

셀 4에서 엑셀 파일을 업로드했다면 이 셀로 지식베이스에 반영합니다.

In [ ]:
os.chdir('/content/insurance-agent')
!python scripts/build_knowledge_from_excel.py --apply --rebuild
print('✅ 지식베이스 + ChromaDB 업데이트 완료')

## 셀 6 — (선택) ChromaDB 벡터 DB 구축

보험 지식베이스를 벡터 DB로 변환합니다 (약 2~5분).  
처음 실행 시 임베딩 모델 (~443MB) 다운로드.

In [ ]:
os.chdir('/content/insurance-agent')
!python scripts/build_vectorstore.py
print('✅ ChromaDB 구축 완료')

## 셀 7 — 서버 실행

**Colab 내장 프록시**로 외부 접속 URL을 생성합니다. (별도 계정/토큰 불필요)  
스트리밍이 안 될 경우 아래 **ngrok 대안** 셀을 사용하세요.

In [ ]:
import os, sys, subprocess, time

PROJECT_DIR  = '/content/insurance-agent'
WEB_APP_PATH = f'{PROJECT_DIR}/web_app.py'

if not os.path.exists(WEB_APP_PATH):
    raise FileNotFoundError(
        f'web_app.py 를 찾을 수 없습니다: {WEB_APP_PATH}\n'
        '셀 2(코드 클론)를 먼저 실행하세요.'
    )

PORT = 5000

# 기존 프로세스 종료 (포트 충돌 방지)
os.system(f'fuser -k {PORT}/tcp 2>/dev/null || true')
time.sleep(1)

# Flask를 별도 subprocess로 실행 — sys.path / 패키지 import 문제 원천 차단
flask_proc = subprocess.Popen(
    [sys.executable, WEB_APP_PATH],
    cwd=PROJECT_DIR,
    env={**os.environ, 'PYTHONPATH': PROJECT_DIR}
)
time.sleep(3)

if flask_proc.poll() is not None:
    raise RuntimeError('Flask 서버가 즉시 종료됐습니다. 로그를 확인하세요.')

print(f'✅ Flask 서버 실행 중 (PID {flask_proc.pid})')

# Colab 내장 프록시로 접속 URL 생성
from google.colab.output import eval_js
public_url = eval_js(f'google.colab.kernel.proxyPort({PORT})')

print('=' * 60)
print('🌐 보험 상담 AI 접속 URL:')
print(f'   {public_url}')
print('=' * 60)
print('⚠️  이 셀을 중단(■)하거나 Colab 탭을 닫으면 서버가 종료됩니다.')

## 셀 7-B — (대안) ngrok으로 실행

Colab 내장 프록시에서 스트리밍(채팅 실시간 출력)이 안 될 경우 사용하세요.  
ngrok 무료 계정 토큰: https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
# ngrok 설치
!pip install -q pyngrok

NGROK_TOKEN = 'YOUR_NGROK_TOKEN'  # ← ngrok 토큰 입력

import os, sys, subprocess, time
from pyngrok import ngrok, conf

conf.get_default().auth_token = NGROK_TOKEN

# 기존 터널 종료
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

PROJECT_DIR  = '/content/insurance-agent'
WEB_APP_PATH = f'{PROJECT_DIR}/web_app.py'
PORT = 5000

# 셀 7을 이미 실행했다면 Flask 가 이미 뜬 상태이므로 연결만
# 아니라면 여기서 직접 실행
try:
    import urllib.request
    urllib.request.urlopen(f'http://localhost:{PORT}/api/status', timeout=2)
    print(f'✅ 기존 Flask 서버 감지 (port {PORT})')
except Exception:
    os.system(f'fuser -k {PORT}/tcp 2>/dev/null || true')
    time.sleep(1)
    subprocess.Popen(
        [sys.executable, WEB_APP_PATH],
        cwd=PROJECT_DIR,
        env={**os.environ, 'PYTHONPATH': PROJECT_DIR}
    )
    time.sleep(3)
    print(f'✅ Flask 서버 시작')

tunnel = ngrok.connect(PORT)
print('=' * 60)
print('🌐 보험 상담 AI 접속 URL (ngrok):')
print(f'   {tunnel.public_url}')
print('=' * 60)

## 참고 — Colab 지원 기능

| 기능 | Colab 지원 | 비고 |
|------|-----------|------|
| GPT-4o 채팅 | ✅ | OpenAI API 키 필요 |
| 보험다모아 엑셀 검색 | ✅ | 셀 4에서 XLS 업로드 후 사용 가능 |
| 실시간 웹 검색 | ✅ | DuckDuckGo (API 키 불필요) |
| FSS API (연금보험) | ✅ | FSS_API_KEY 설정 시 |
| ChromaDB RAG | ✅ | 셀 6 실행 시 |
| 신용점수 포트폴리오 탭 | ✅ | 완전 지원 |
| 건강위험 포트폴리오 탭 | ✅ | 로지스틱 모델 기반 위험 예측 |
| **🏆 대회 데모 탭** | ✅ | **이노베이션 존 10시나리오 완전 지원** |
| 보험다모아 실시간 스크래핑 | ⚠️ | CDP 모드 불가 |
| NICE/KCB 신용점수 CDP 조회 | ❌ | 로컬 Chrome 필요 |

### 대회 데모 탭 — 이노베이션 존 10시나리오

| # | 영역 | 시나리오 | 활용 데이터 |
|---|------|----------|-------------|
| 1 | 보험 | 암 완치자 인수 심사 | RGST(암등록 261만건) · DEATH · G1E |
| 2 | 보험 | AI 저위험군 보험료 할인 | G1E(건강검진 1657만건) · 광주TP DICOM |
| 3 | 보험 | 미세 영상 소견자 노-할증 | 광주TP DICOM/JPG · T400(상병) |
| 4 | 보험 | 동적 보험료 캐시백 | cdw_lflg(라이프로그) · cdw_psmn_vtls |
| 5 | 보험 | 맞춤형 유병자 요율 | T200~T530(상병) · BFC(보험료분위) |
| 6 | 금융 | 씬파일러 Health-Credit 신용평가 | G1E · cdw_psmn_vtls · BFC · CB 신용DB |
| 7 | 금융 | 소상공인 건강 지속가능성 대출 | 광주TP CDW · RGST · 매출 DB |
| 8 | 금융 | 유병자·고령층 렌탈 금융 승인 | cdw_ptn_hli · DEATH · RGST |
| 9 | 위험관리 | 미시 징후 사전 케어 암 중증화 차단 | 광주TP DICOM · T400 · 보험사 지급 DB |
| 10 | 위험관리 | 중증 질환 전환 예측 부실률 차단 | cdw_bacm_sofa_isp · RGST · DEATH |

### 데이터 구성

| 구분 | 내용 | 출처 |
|------|------|------|
| 암종별 발생률 | 연령대·성별·암종별 10만명당 발생률 | 국립암센터 **2022 암등록통계** 공식 발표 |
| 5년 생존율 | 암종별 5년 상대 생존율 | 국립암센터 **2022 암등록통계** 공식 발표 |
| 암 위험 배율 | 흡연·비만 등 위험인자별 상대위험도 | **공개 역학 논문** (흡연→폐암 10배 등) |
| 보험료 분위 | BFC 10분위 소득·보험료 기준 | **NHIS 건강보험료 10분위** 기준 추정 |
| 폐 소결절 악성화율 | 크기별 악성 전환 확률 | **Fleischner Society** 임상 가이드라인 |
| 암 완치자 재발률 | 암종·병기·치료법별 10년 재발률 | 의학 교과서·임상 논문 수치 |
| 건강위험 예측 모델 | 로지스틱 회귀 (AUC 0.78, n=32,000) | NHIS 건강검진 **스키마 기반 합성 데이터** 학습 |
| 페르소나 | 박민준·이수진 등 10명 | 대회 시나리오 기반 수동 설계 가상 인물 |

> 이노베이션 존 **실데이터 접근 후** `data/innovation_zone_ref.py` 수치와 `data/risk_model_params.json` 모델 파라미터만 교체하면 나머지 로직은 그대로 동작합니다.

### Colab 세션 유지 팁
브라우저 콘솔(F12)에서 아래 코드를 실행하면 자동 연결 유지됩니다:
```javascript
function KeepAlive() {
  document.querySelector('#top-toolbar .yes-button')?.click();
  setTimeout(KeepAlive, 60000);
}
KeepAlive();
```
